In [1]:
import sys
from pathlib import Path
import pandas as pd
from faker import Faker
import hashlib

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.aih_privacy.config import DATA_PROCESSED_DIR
from src.aih_privacy.datasets.sisfall.subjects import load_subjects


In [2]:
subjects_df = load_subjects().copy()
print("Loaded subjects_df:", subjects_df.shape)
subjects_df.head()

Loaded subjects_df: (38, 6)


,subject_id,age,height_cm,weight_kg,gender,age_group
0,SA01,26,165,53.0,F,SA
1,SA02,23,176,58.5,M,SA
2,SA03,19,156,48.0,F,SA
3,SA04,23,170,72.0,M,SA
4,SA05,22,172,69.5,M,SA


In [3]:
PSEUDONYM_SALT = "aih-sisfall-v1"  # mantém fixo e não metas no repositório público se for sensível

def make_pid(subject_id: str, salt: str = PSEUDONYM_SALT) -> str:
    return hashlib.sha256(f"{salt}:{subject_id}".encode()).hexdigest()[:16]

subjects_df["pid"] = subjects_df["subject_id"].apply(make_pid)
subjects_df[["subject_id", "age_group", "pid"]].head()


,subject_id,age_group,pid
0,SA01,SA,8b39978f3bf493e0
1,SA02,SA,8cca10112272b7d0
2,SA03,SA,5a58367b9ab21a3f
3,SA04,SA,ac62c148b01bdf2b
4,SA05,SA,144051cc52779481


In [4]:
fake = Faker()
fake.seed_instance(42)  # reprodutível

subjects_df["synthetic_name"] = [fake.name() for _ in range(len(subjects_df))]
subjects_df["synthetic_address"] = [fake.address().replace("\n", ", ") for _ in range(len(subjects_df))]
subjects_df["synthetic_phone"] = [fake.phone_number() for _ in range(len(subjects_df))]
subjects_df["synthetic_patient_id"] = [fake.bothify(text="??-########") for _ in range(len(subjects_df))]

subjects_df.head()

,subject_id,age,height_cm,weight_kg,gender,age_group,pid,synthetic_name,synthetic_address,synthetic_phone,synthetic_patient_id
0,SA01,26,165,53.0,F,SA,8b39978f3bf493e0,Allison Hill,"893 Nathaniel Estates Apt. 957, North Sarahpor...",631-335-1823x374,Xx-02681177
1,SA02,23,176,58.5,M,SA,8cca10112272b7d0,Noah Rhodes,"71822 Arroyo Expressway, Allisonchester, IL 71187",643.252.4082,UY-89178390
2,SA03,19,156,48.0,F,SA,5a58367b9ab21a3f,Angie Henderson,"465 Lam Mission, East Jeffreymouth, AK 77611",9849271094,Rg-84700766
3,SA04,23,170,72.0,M,SA,ac62c148b01bdf2b,Daniel Wagner,"10310 Jones Freeway, Elizabethborough, ND 17843",252.404.7116x719,ir-77115921
4,SA05,22,172,69.5,M,SA,144051cc52779481,Cristian Santos,"76311 Gomez Loop Suite 010, Chandlerville, IA ...",(594)813-1869,DG-99856984


In [5]:
out_dir = DATA_PROCESSED_DIR / "sisfall"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "subjects_identity.parquet"
subjects_df.to_parquet(out_path, index=False)

print("Saved:", out_path)

Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\subjects_identity.parquet
